In [ ]:
import os

import coiled
import matplotlib.pyplot as plt
import xarray as xr
from frisky import hijack

from saidownscale.qa_flags import (
    DIR_QA_FLAG_CONSTANT_INPUTS,
    calculate_thresholds,
    discover_leaves,
    flag_global_exceedances,
    flag_outliers,
    flag_rsds_above_max,
    run_flag_loop,
)

os.environ["FRISKY_SUMMARY"] = "off"

In [ ]:
VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds"]
METHODS = ["bcsd", "qdmsd"]
GCMS = ["CESM2-WACCM6", "UKESM1-1-LL"]

BRANCH = "v1.0.0"
ROOT_DIR = "s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/"
STORE_SUBSET_ID = "global"

PLOT_FLAG_MAPS = True
BUCKET = "carbonplan-srm"
PREFIX = "scratch/output/qa-intermediate-flags-v1.0.0-qa-run2"

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

## Set up cluster

In [ ]:
cluster = coiled.Cluster(
    name="srm-qaqc-flags-nrh",
    region="us-west-2",
    n_workers=12,
    worker_vm_types=["m8gn.xlarge"],
    scheduler_vm_types="c8g.xlarge",
    spot_policy="spot_with_fallback",
    use_best_zone=True,
    tags={"Project": "SRM"},
    worker_options={"nthreads": 8},
    environ={"ZARR_ASYNC__CONCURRENCY": "128"},
)

client = hijack(cluster.get_client())
client

# Option 1: call `run_step2` function (goes through all steps at once, for both downscaled and debiased coarse)

In [ ]:
from saidownscale.qa_flags import run_step2

In [ ]:
run_step2(
    variables=VARIABLES,
    gcms=GCMS,
    methods=METHODS,
    branch=BRANCH,
    root_dir=ROOT_DIR,
    store_subset_id=STORE_SUBSET_ID,
    bucket=BUCKET,
    prefix=PREFIX,
    plot_flag_maps=True,
    verbose=True,
    mode="debiased_coarse_only",
)

# Option 2: go through each step one at a time (just for downscaled, not including debiased coarse in this demo)

In [ ]:
IS_DOWNSCALED = True

### A. Define what data arrays exist to traverse

In [ ]:
[
    trees,
    tags,
    gcms_np,
    scenarios_np,
    variables_np,
    tags_np,
    methods_np,
    debiased_coarse_flags_np,
] = discover_leaves(
    gcms=GCMS,
    branch=BRANCH,
    root_dir=ROOT_DIR,
    store_subset_id=STORE_SUBSET_ID,
    is_downscaled=IS_DOWNSCALED,
)

### B. Generate individual time-varying QA flags

#### 1. global exceedances

In [ ]:
%%time
run_flag_loop(
    tags=tags,
    trees=trees,
    bucket=BUCKET,
    prefix=PREFIX,
    flag_name="outside_global_plausible_range",
    compute_flag=lambda da, var: flag_global_exceedances(da=da, var=var),
    write_mode="a",
    is_downscaled=IS_DOWNSCALED,
)

# with plotting ~14.5 mins

#### 2. Outliers based on observations

##### 2a. Calculate outlier bounds

In [ ]:
# Loading thresholds output from step 1 notebook
store = DIR_QA_FLAG_CONSTANT_INPUTS + "doy_obs_thresholds_global.zarr"
combined = xr.open_zarr(store)

In [ ]:
def _split(ds, suffix):
    names = [v for v in ds.data_vars if v.endswith(suffix)]
    return ds[names].rename({v: v[: -len(suffix)] for v in names})


obs_max = _split(combined, "_max")
obs_min = _split(combined, "_min")
obs_max_std = _split(combined, "_max_std")
obs_min_std = _split(combined, "_min_std")

In [ ]:
[outlier_thresh_low, outlier_thresh_high] = calculate_thresholds(
    obs_max, obs_min, obs_max_std, obs_min_std
)

##### 2b. Traverse dataset and save flags

In [ ]:
outlier_thresh_low_annual = outlier_thresh_low.min(dim="dayofyear")
outlier_thresh_high_annual = outlier_thresh_high.max(dim="dayofyear")

In [ ]:
run_flag_loop(
    tags=tags,
    trees=trees,
    flag_name="annual_outlier_flag",
    compute_flag=lambda da, var: flag_outliers(
        da=da,
        outlier_thresh_low=outlier_thresh_low_annual[var],
        outlier_thresh_high=outlier_thresh_high_annual[var],
        timescale="annual",
    ),
    bucket=BUCKET,
    prefix=PREFIX,
    write_mode="a",
    is_downscaled=IS_DOWNSCALED,
)

#### 3. rsds-specific latitude check

In [ ]:
def load_rsds_lims(
    key: str = "zonal_doy_max_rsds",
    fpath: str = DIR_QA_FLAG_CONSTANT_INPUTS + "zonal_doy_max_rsds.zarr",
) -> xr.DataArray:
    return xr.open_zarr(fpath, group=key)["data"].load()

In [ ]:
zonal_doy_max_rsds = load_rsds_lims()

In [ ]:
run_flag_loop(
    tags=tags,
    trees=trees,
    flag_name="rsds_max_exceeded",
    compute_flag=lambda da, var: flag_rsds_above_max(da=da, zonal_doy_max_rsds=zonal_doy_max_rsds),
    var_filter=["rsds"],
    bucket=BUCKET,
    prefix=PREFIX,
    write_mode="a",
    is_downscaled=IS_DOWNSCALED,
)

#### 4. temperature inconsistencies

In [ ]:
from saidownscale.qa_flags import run_flag_loop_temperature_inconsistencies

In [ ]:
run_flag_loop_temperature_inconsistencies(
    tags=tags,
    trees=trees,
    flag_name="temperature_inconsistency",
    bucket=BUCKET,
    prefix=PREFIX,
    is_downscaled=IS_DOWNSCALED,
    plot=True,
    save_plots=False,
)

### C. Generate flags that are constant over time

In [ ]:
from saidownscale.qa_flags import (
    calculate_trend_distortion_flags,
)

In [ ]:
calculate_trend_distortion_flags(
    trees=trees,
    gcms=GCMS,
    variables=["pr"],
    methods=METHODS,
    tags_np=tags_np,
    gcms_np=gcms_np,
    scenarios_np=scenarios_np,
    variables_np=variables_np,
    methods_np=methods_np,
    bucket=BUCKET,
    prefix=PREFIX,
    is_downscaled=IS_DOWNSCALED,
    plot=True,
    distortion_flag_calculation_type="v2",
    save_plots=False,
)

In [ ]:
if cluster is not None:
    cluster.shutdown()
else:
    print("no cluster was created (cached run); nothing to shut down")

### D. Read in flags

In [ ]:
from saidownscale.qa_flags import get_intermediate_flags

gcm = "CESM2-WACCM6"
var = "rsds"
scenario = "ssp245"
ens = "008"
method = "qdmsd"

tag = f"{gcm}_{var}_{scenario}_{ens}_{method}"
flag_ds = get_intermediate_flags(tag=tag, bucket=BUCKET, prefix=PREFIX)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2)
flag_frac = flag_ds["outside_global_plausible_range"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[0, 0])
flag_frac = flag_ds["rsds_max_exceeded"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[0, 1])
flag_frac = flag_ds["annual_outlier_flag"].mean(dim="time")
flag_frac.where(flag_frac > 0).plot(ax=axes[1, 0])
plt.tight_layout()

In [ ]:
flag_ds["rsds_max_exceeded"].sum(dim="time").plot()
plt.plot([28.5], [-29.6], "xr")